# 05 — Synthesis & Audit: Current State vs Publication Plan

> **Scope of this notebook.** This is an engineering audit of the `02_simula/` codebase as of 2026-04-23 against the plan in `01_article/01_publication_roadmap.md` (and `04_rk4_working_plan.md`). It has four sections, matching the user's audit request:
>
> 1. **Synthesis of the current state** — what is built and functional.
> 2. **Parameters & outputs** — everything that is hardcoded, the values, and what the pipeline can emit.
> 3. **Error checking & best practices** — bugs, data-flow issues, signal-processing caveats.
> 4. **Gap analysis & redirection** — a concrete step-by-step plan to close the gap to a publishable study.

Format: mostly markdown. Two light code cells at the end print the current `constants.py` values and a minimal sanity-check so this notebook can double as a pinned reference when future notebooks are refactored.

## 1. Current state synthesis

### 1.1 Pipeline shape

The code forms a **serial pipeline, glued through a single file (`sinha_rotor.toml`)**, not through importable Python modules:

```
00_sinha_rotor.ipynb        builds rotor, calibrates kxx to f1≈27.50 Hz → writes sinha_rotor.toml
          │
          ▼
sinha_rotor.toml            frozen FE model
          │
          ▼  (each downstream notebook `rs.Rotor.load("sinha_rotor.toml")`)
01_sinha_rotor_modal.ipynb  modal / unbalance / Bode / deflected shapes
01_unbalance.ipynb          explanatory: why Ω=0 is singular, Bode with Ω=[1e-3, 300]
02_sinha_unbalance_phase_crack.ipynb  Gasch crack at depth=0.5, sweep unbalance phase 0°–345°/15°
03_sinha_crack_model_comparison.ipynb Mayes vs Gasch (Flex* disabled) × depth ∈ {0.1 … 0.5}
04_sinha_fault_analysis.ipynb         healthy / crack / misalignment integrative comparison
```

Shared defaults live in `constants.py`; one-off choices (phase grids, depth grids, plot depths, model subsets) stay notebook-local. There is no script entry-point, no DoE driver, and no persisted results (every results dict only lives inside its notebook kernel).

### 1.2 Rotordynamic modeling — what is operational

| Capability | Where | Status |
|---|---|---|
| 13-node / 12-element beam-shaft build (Sinha geometry) | `00_sinha_rotor.ipynb` | ✅ operational |
| Disk element from geometry at node 6 (265 mm) | `00_sinha_rotor.ipynb` | ✅ |
| Isotropic bearings at nodes 1 (20 mm) and 11 (510 mm) | `00_sinha_rotor.ipynb` | ✅ |
| Bearing stiffness calibration via `scipy.optimize.brentq` on `f1 − 27.50 Hz` | `00_sinha_rotor.ipynb` | ✅ |
| Campbell diagram 0–5000 rpm (post-processed to Hz) | `00_sinha_rotor.ipynb` | ✅ |
| Modal analysis (undamped wn, damped wd, damping ratios) at Ω = 100 rad/s | `01_sinha_rotor_modal.ipynb` | ✅ |
| Unbalance response (Bode, deflected shapes) | `01_sinha_rotor_modal.ipynb`, `01_unbalance.ipynb` | ✅ (non-zero speed grid documented) |
| Breathing-crack time response via `rotor.run_crack` (Mayes, Gasch) | `02`, `03`, `04` | ✅ |
| Parallel flexible-coupling misalignment via `rotor.run_misalignment` | `04` | ✅ |
| Manual healthy baseline with `run_time_response` and a user-built unbalance force vector | `04` | ✅ |
| Flex Open / Flex Breathing crack models | `03` config says available | ❌ disabled (`CRACK_MODELS = ["Mayes", "Gasch"]`) |

### 1.3 Signal-processing stack — what exists and what does not

| Technique | Present? | Notes |
|---|---|---|
| 1-sided amplitude FFT via `np.fft.rfft`, normalized `2/N` | ✅ (02, 03) | No window, no detrend, nearest-bin peak pick |
| `plot_dfft` via ROSS (uses its own DFFT internally) | ✅ (04) | Black-boxed; used for FFT panel of the integrative figure |
| Orbit plots at probe / disk nodes | ✅ (02, 03) | Steady-state half of the time record |
| Harmonic-amplitude extraction (1X/2X/3X) by nearest-bin pick | ✅ (02, 03) | `get_harmonic_amplitude()` (defined twice, identical) |
| 2X/1X, 3X/1X harmonic-ratio diagnostics | ✅ (02, 03) | Diagnostic plotted; no threshold or classifier |
| Polar / grouped-bar harmonic-vs-phase views | ✅ (02) | |
| Stiffness breathing law plot `K(θ)/K_intact` | ✅ (03) | Mayes vs Gasch (Flex models excluded by design) |
| **Bispectrum** `B(f1, f2) = E[X(f1)X(f2)X*(f1+f2)]` | ❌ | Mentioned in markdown only (end of `04`, and in roadmap Appendix A.2) |
| **Bicoherence** `b²(f1,f2)` | ❌ | Not implemented |
| **Trispectrum** | ❌ | Not implemented |
| Scalar HOS indicators (BPR, BCS, bispectral entropy, biphase) | ❌ | Not implemented; not even a stub |
| PSD via Welch (scipy.signal.welch) | ❌ | Only raw amplitude spectrum is used |
| Order tracking / angle-domain resampling | ❌ | No keyphasor signal, no resample |
| Windowing (Hann / Hamming / Kaiser) with amplitude or ENBW correction | ❌ | Rectangular window throughout |
| Detrending before FFT | ❌ | Not applied in `02`/`03` (ROSS's `plot_dfft` may do so internally in `04`) |

**Headline:** The rotordynamic half of the thesis is essentially in place; the HOS half — which is the *core novelty claim* of the research — has **zero lines of implementation**. Everything in `02`–`04` is still traditional FFT/harmonic-ratio work.

### 1.4 What the existing code can currently produce (outputs inventory)

Per notebook, the artifacts that are actually rendered today:

| Notebook | Plots / tables | Numerical outputs |
|---|---|---|
| `00_sinha_rotor` | Rotor layout; Campbell 0–5000 rpm (Hz) | `k_opt` (N/m), `modal.wn[0..2]`, saved `sinha_rotor.toml` |
| `01_sinha_rotor_modal` | Mode 3D plot; deflected shape at 650 & 750 rpm; Bode at probe | First 12 wn / wd (Hz), damping ratios |
| `01_unbalance` | Bode without the Ω=0 singular bin | Count of warnings with/without `1e-3` grid start |
| `02_sinha_unbalance_phase_crack` | 24 orbit subplots × 2 speeds; 24 FFT subplots × 2 speeds; overlaid spectra; grouped-bar harmonic amplitudes; polar plots; steady-state time waveforms; disk-node orbit grids; 2X/1X, 3X/1X ratio curves | `dfs[speed_rpm]` DataFrames (columns: `phase_deg`, `1X`, `2X`, `3X`, and derived ratios) |
| `03_sinha_crack_model_comparison` | 2 models × 5 depths orbits; overlaid FFTs @ depth 0.5; 1X/2X/3X line plots; 2X/1X ratio vs depth; waveforms; normalized stiffness curves | `harmonics_tables[speed_rpm][model]` DataFrames |
| `04_sinha_fault_analysis` | Time response for healthy/crack/mis @ pair of speeds; 7-row integrative time + FFT comparison figure | None persisted — everything is in-memory |

**No artifact is written to disk.** Figures are displayed inline; DataFrames are printed to stdout and discarded. There is no `results/`, no HDF5, no parquet, no CSV.

## 2. Parameters & generated quantities

### 2.1 Shared constants (`constants.py` — canonical)

Geometry / topology:
- `BEARING_1_NODE = 1` (20 mm), `BEARING_2_NODE = 11` (510 mm), `DISK_NODE = 6` (265 mm)
- `CRACK_NODE = DISK_NODE + 1 = 7` (315 mm), `PROBE_NODE = BEARING_2_NODE − 1 = 10` (490 mm)

Forcing:
- `UNB_MAG = Q_(2e-4, "kg·m")`
- `UNB_PHASE = Q_(4π/3 rad)` ≈ 240° — canonical (overrides the inline 3π/4 that `04` used pre-Sprint-01)

Speeds:
- Crack campaign: `SPEED_CRACK_0 = 650 rpm` (10.833 Hz), `SPEED_CRACK_1 = 750 rpm` (12.5 Hz)
- Misalignment campaign: `SPEED_MIS_0 = 750 rpm`, `SPEED_MIS_1 = 900 rpm` (15.0 Hz)
- `SPEEDS = [SPEED_CRACK_0, SPEED_CRACK_1]` (alias)

Fault severities:
- `CRACK_RATIO = 0.5` (default; `03` sweeps {0.1, 0.2, 0.3, 0.4, 0.5}, Mayes/Gasch cap)
- `MIS_X = 1.0e-3 m`, `MIS_Y = 0.5e-3 m` (parallel offsets for flex coupling)

Time / frequency grids:
- `DT = 1e-3 s` → **`FS_SIM_HZ = 1000 Hz`**, Nyquist = 500 Hz
- `T = np.arange(0.0, 2.0, DT)` → 2000 samples, 2.0 s record
- `FREQ_RANGE = Q_((0, 200), "Hz")` for plotting

Sinha (2007) HOS handoff (declared but **not yet consumed** by any code):
- `SINHA_FS_HZ = 2560`, `SINHA_AA_CUTOFF_HZ = 1000`
- `SINHA_HOS_DF_HZ = 1.25`, `SINHA_HOS_N_SEGMENTS = 50`, `SINHA_HOS_OVERLAP = 0.5`

### 2.2 Notebook-local knobs

| Knob | Where | Value |
|---|---|---|
| `kxx0` seed for Brent calibration | `00` | `2.0e6 N/m` |
| `cxx0` / modal bracket | `00` | `50.0 N·s/m` / `[1e4, 1e6]` |
| Campbell speed grid | `00` | `np.linspace(0, 5000, 30) RPM`, `frequencies=4` |
| Modal speed / mode count | `01_sinha_rotor_modal` | `Q_(100, "rad/s")`, `num_modes=12` |
| Bode speed grid | `01`, `01_unbalance` | `np.linspace(1e-3, 300, 1000) rad/s` |
| Unbalance-phase sweep | `02` | `np.arange(0, 360, 15)°` (24 points) |
| `model_reduction` (`run_crack`) | `02`, `03` | `{"num_modes": 12}` |
| Crack models in comparison | `03` | `["Mayes", "Gasch"]` (Flex Open/Breathing commented out) |
| Depth-ratio sweep | `03` | `[0.1, 0.2, 0.3, 0.4, 0.5]` |
| `PLOT_DEPTH` for orbit/FFT panels | `03` | `0.5` |
| `CROSS_DIVISIONS` (Flex Breathing only) | `03` | `10` |
| Steady-state start index | `02`, `03` | `int(len(T) * 0.5)` (= 1000, i.e. discard 1 s of 2 s) |
| Coupling radial stiffness / bending stiffness (`run_misalignment`) | `04` | `radial_stiffness=4e4 N/m`, `bending_stiffness=3.8e4 N·m/rad` |
| Misalignment method | `04` | `method="newmark"` |
| `n_revs` window for waveform compare | `02`, `03` | `2` revolutions |

## 3. Error check & best-practices audit

Findings are ranked by **severity** for the publishable-study goal. Each entry cites the file so the fix is easy to pinpoint.

### 3.1 Critical issues (block publishability)

**C1 — The HOS pipeline does not exist.** The entire claim of the thesis ("numerical HOS for crack vs misalignment") has no code. `02_sinha_unbalance_phase_crack.ipynb` even has a `# TODO` above its FFT block asking whether that is even the power FFT (it is not — it is an amplitude spectrum). Everything downstream (indicators, discrimination, sensitivity curves) is therefore also missing.

**C2 — Record length is insufficient for Sinha-matched HOS.** `DT=1e-3 s`, `T=2.0 s`, steady window = 1.0 s ⇒ **1000 samples** per case after transient cut-off.

- Sinha's Welch-style HOS settings (in `constants.py`): `fs=2560 Hz`, `df=1.25 Hz`, `N_seg=50`, `overlap=0.5`. That requires `Nfft = fs/df = 2048` samples per segment, total ≈ `Nfft · (1 + 0.5·(N_seg−1)) ≈ 52 k samples` ≈ **20.3 s at 2560 Hz**.
- The current data supports **one 1024-point segment, period**. Variance of any bispectral estimator will be enormous; bicoherence will saturate at 1 everywhere that has any signal.
- Fix: **raise `DT` to 1/2560 s** and **lengthen `T` to ≥ 20–30 s** for the HOS cases (healthy/crack/misalignment at each speed). Keep a separate short-record preset for the existing orbit / phase-sweep notebooks.

**C3 — Simulation fs = 1000 Hz < Sinha fs = 2560 Hz.** `FS_SIM_HZ = 1000` means Nyquist = 500 Hz; `SINHA_AA_CUTOFF_HZ = 1000` is physically *above* Nyquist, so the constant is currently self-contradictory. Either bump `FS_SIM_HZ` to 2560 Hz or explicitly decouple `fs_sim` from `fs_acq` and resample in post-processing. The `run_crack` / `run_misalignment` integrators will handle a finer `DT` fine.

**C4 — No shared speed grid across fault types.** Crack uses {650, 750} rpm, misalignment uses {750, 900} rpm. Only 750 rpm is shared. A fault-discrimination study cannot compare faults at *different speeds* because 2X(crack @650) = 21.7 Hz while 2X(mis @900) = 30 Hz — almost any indicator will track the speed, not the fault. Either align the two campaigns on `{SPEED_CRACK_0, SPEED_CRACK_1}` or introduce a proper DoE (Section 4).

### 3.2 High-severity issues (produce noisy or misleading results)

**H1 — `04` reuses the wrong healthy baseline for the 900 rpm misalignment case.** In `04_sinha_fault_analysis.ipynb` the comparison `cases` list contains `{label: "Healthy @ 900 rpm", result: healthy_1, speed: SPEED_MIS_0}` — but `healthy_1` was simulated at 750 rpm and `SPEED_MIS_0` is 750 rpm while `SPEED_MIS_1 = 900 rpm` has *no* paired baseline. The integrative figure therefore compares misalignment-at-900 rpm against unbalance-at-750 rpm. **Bug, not a physics nuance.** Fix: add `healthy_mis_1 = run_healthy_baseline(..., SPEED_MIS_1, ...)` and pair properly.

**H2 — FFT without window and without detrend.** `02` / `03` compute `np.fft.rfft(x_ss)` on a rectangular window. Because `steady_start = len(T)//2 = 1.0 s` does not contain an integer number of revolutions at 650 rpm (10.833 rev/s → 10.833 revolutions in 1 s, i.e. non-integer), the 1X/2X/3X peaks *do not land on bins*. Symptoms:
- Up to **−3.92 dB scalloping loss** on peak-amplitude estimates (rectangular window worst case).
- Leakage into neighboring bins that `get_harmonic_amplitude` picks up with `np.argmin(|freqs − target|)`.

Fix choices, in order of effort:
1. **Make the record an integer number of revolutions at each speed** (trim the record in post; the crack sim itself does not change).
2. Apply a **Hann window** to the segment and correct amplitude by `2 / sum(window)`, or use parabolic-interpolation peak picking on the windowed spectrum.
3. Move to **order tracking** (angle-domain resample using the rotation phase) — this is the right long-run answer for a rotordynamics paper.

**H3 — Amplitude normalization mixes conventions.** `fft_amp = 2.0 / N * np.abs(np.fft.rfft(...))` is valid for a rectangular window and an integer-period sinusoid. With any window or non-integer period it is neither a correct amplitude spectrum nor a correct PSD. Decide which one is wanted (probably **PSD via `scipy.signal.welch`** for the HOS preamble, and a windowed amplitude spectrum for the 1X/2X harmonic-ratio diagnostic) and keep one per purpose.

**H4 — Modal reduction to 12 modes may truncate HOS content.** `run_crack(..., model_reduction={"num_modes": 12})` is used in every fault sim. Modal truncation is fine for low-frequency orbits but HOS bins at (2X, 2X), (2X, 3X), (3X, 3X) for 900 rpm reach 45–90 Hz — near the higher modes of a 13-node rotor — and any energy in those bins attributable to modal residuals will be dropped. Before the HOS campaign, rerun one case with `num_modes` ∈ {12, 24, 36} and confirm the HOS magnitudes at the key bins have converged.

### 3.3 Medium-severity issues (correctness or maintainability)

**M1 — `01_sinha_rotor_modal.ipynb` passes a pint `Quantity` as the unbalance phase.** `unbalance_phase=[UNB_PHASE]` in the `run_unbalance_response` call — should be `.to("rad").m` for parity with `02`/`03`/`04`. Silent coercion risk depending on ROSS's Quantity handling.

**M2 — `get_harmonic_amplitude` is defined in both `02` and `03` with identical bodies.** It reads `steady_start` from the enclosing notebook scope (a closure-over-globals), so copying it into another notebook without also copying `steady_start` breaks silently. Move it to a small module (`02_simula/signal_utils.py`) together with a windowed, interpolated version for Section 4.2.

**M3 — Manual healthy-baseline forcing in `04` mixes solver paths with the crack/misalignment cases.** `run_healthy_baseline` hand-builds an unbalance force vector and calls `rotor.run_time_response`. The crack/misalignment cases call `rotor.run_crack` / `rotor.run_misalignment`, which apply unbalance *internally* via ROSS's own force assembly. Integrator, transient length, and initial conditions can differ subtly. For a fair apples-to-apples comparison, drive every case through the same top-level solver (either always `run_time_response` with a user-supplied force, or always `run_crack` / `run_misalignment` with `depth_ratio=0` or `mis_distance=0`).

**M4 — `fix_dof` expression is copy-pasted and undocumented.** `fix_dof = (PROBE_NODE - nodes[-1] - 1) * ndof // 2 if PROBE_NODE in link_nodes else 0` — this only fires when `PROBE_NODE in link_nodes`, which is empty for the current Sinha rotor, so it is benign today. But the math (subtract half an `ndof`) is not obviously right for 4- or 6-DOF-per-node shafts; it will break quietly the day a `link_node` is added. Add a test (even a simple assert) or replace with ROSS's own DOF resolver.

**M5 — `steady_start = int(len(T) * 0.5)` is an untested heuristic.** For heavily damped modes, 1 s is plenty; for lightly damped near-critical runs it may not be. There is no check that the transient has actually decayed (e.g., cross-correlation of first/second halves of the retained window, or energy envelope). Before the HOS campaign, add a settling check or use a longer run with a dynamically chosen steady window.

**M6 — No random seed, no per-run metadata.** Results are *deterministic* today (no noise), but the moment measurement noise is added (as the roadmap R6 mitigation asks), you will need a seed. Similarly nothing records `ross.__version__`, `depth_ratio`, `phase`, `speed`, and `num_modes` alongside the arrays — there is no audit trail for a figure.

### 3.4 Low-severity / stylistic

**L1 — Roadmap Appendix A.2 bispectrum reference code recomputes the full FFT per `(f1,f2)` pair.** `X3 = np.fft.fft(segment)[f3]` inside the inner loop is `O(N_f² · N log N)` per segment. The same appendix hints at `nfft=512` as a feasibility default. Before using it: hoist `X_full = np.fft.fft(segment)` out of the inner loop (the version in `03_ross_agent_prompt.md` already does).

**L2 — Two identical `get_harmonic_amplitude` defs, two identical DOF-extraction blocks.** Consolidate into a utility module.

**L3 — `MIS_X`, `MIS_Y` are not used in `run_misalignment`.** `04` passes the numbers inline (`mis_distance_x=1.0e-3, mis_distance_y=0.5e-3`) rather than the constants. Single-source-of-truth regression.

**L4 — Two `Mayes` docstrings / narratives describe `run_crack` as using Mayes, while the actual call uses `crack_model="Gasch"` (notebook `04`).** Text and call drifted; rewrite the theory block to match what is executed, or switch the executed call to Mayes.

## 4. Gap analysis & step-by-step redirection

Against the phases in `01_article/01_publication_roadmap.md §5` and the MVP in `§7.1`:

| Roadmap phase | Status | Gap |
|---|---|---|
| Phase 1 — Foundation | 🟡 partial | Rotordynamics side is OK; HOS-theory / bispectrum-estimation literature distillation is not captured in-repo |
| Phase 2 — Model development | 🟢 mostly done | FE model calibrated; Mayes+Gasch and misalignment sims running; Flex-Open/Breathing and proper validation plots still to do |
| Phase 3 — Simulation campaign | 🔴 **not started** | No DoE script, no persisted dataset, no shared speed / fault-severity grid |
| Phase 4 — HOS analysis | 🔴 **not started** | Zero bispectrum / bicoherence / trispectrum code; zero scalar indicators; zero discrimination analysis |
| Phase 5 — Paper writing | 🔴 not started | Roadmap / executive summary / RK4 plan exist; article itself does not |

The existing notebooks are a very solid **Phase 2** artifact. Everything you have *planned* to publish is stuck at the Phase 3 / Phase 4 boundary.

### 4.1 Redirection roadmap (concrete, ordered)

Think of the next four weeks as four focused deliverables. Each one closes one of the blockers above.

---

#### Step 1 — Retire the sample-rate / record-length risk (≈ 1 day)

*Fixes C2, C3, part of H4.*

1. In `constants.py`, add:
   - `DT_SIM = 1.0 / 2560`  (so `FS_SIM_HZ = 2560`, matching `SINHA_FS_HZ`)
   - `T_LONG = np.arange(0.0, 25.0, DT_SIM)`  (25 s record for HOS cases)
   - Keep the existing short `T` under a new name (`T_SHORT`) for the orbit/phase-sweep notebooks — they do not need long data.
2. Update the `constants.py` provenance docstring to reflect the two-grid decision.
3. Spot-check: rerun one Gasch @ 650 rpm, depth 0.5 under the new grid. Confirm the orbit at the probe node matches the existing figure within ~1 µm peak-to-peak (sanity that the integrator is converged).

**Stop criterion:** one cell in `03` or `04` that loads `T_LONG`, runs a crack case, and produces a time trace whose 2X/1X ratio is within 5% of the current value.

---

#### Step 2 — Extract a `signal_utils.py` and write the HOS core (≈ 2–3 days)

*Fixes C1, H2, H3, M2, L1, L2. This is where the thesis becomes a thesis.*

Create `02_simula/signal_utils.py` with:

```python
def window_and_detrend(x, window="hann"): ...
def amplitude_spectrum(x, fs, window="hann"): ...        # H-corrected single-sided
def psd_welch(x, fs, nperseg, noverlap, window="hann"): ...
def bispectrum(x, fs, nfft, noverlap, window="hann"): ...     # direct method, segment-averaged
def bicoherence(x, fs, nfft, noverlap, window="hann"): ...    # |B|² / (P12 · P3), with small-denom guard
def harmonic_amplitude(x, fs, f_target, window="hann"): ...   # parabolic-interp peak pick
```

Mathematical spec (use these definitions; cite in the thesis Methods):

- Bispectrum (direct, segment-averaged):
  $$\hat B(f_1, f_2) = \frac{1}{K} \sum_{k=1}^{K} X_k(f_1)\, X_k(f_2)\, X_k^{*}(f_1+f_2)$$
- Bicoherence (normalized, [0, 1]):
  $$\hat b^2(f_1, f_2) = \frac{|\hat B(f_1, f_2)|^2}{\big\langle |X(f_1) X(f_2)|^2 \big\rangle \cdot \big\langle |X(f_1+f_2)|^2 \big\rangle}$$
- Biphase: `angle(B(f1, f2))`.

Implementation discipline:
- **Hoist `X_full = np.fft.fft(windowed_segment)` out of the `(f1,f2)` double loop.** The `03_ross_agent_prompt.md` version does; the Appendix A.2 version does not — reuse the former.
- Apply a Hann window and correct amplitudes by `2 / sum(w)`; correct bispectral normalization by `sum(w³)` (Kim & Powers 1979 convention).
- Add a synthetic-signal unit test: a signal `cos(2πft) + 0.5 cos(4πft + φ) + 0.5 cos(6πft + φ)` with enforced quadratic phase coupling should show `b²(f,f) ≈ 1`, `b²(f,2f) ≈ 1`, and `b² ≈ 0` elsewhere. **This is the single most important test in the whole thesis** — without it no reviewer will trust any bispectrum plot.

Write a tiny driver notebook (`06_hos_validation.ipynb`) that runs this test and one real simulation (Gasch crack @ 650 rpm, depth 0.5) side-by-side. This retires the critical risk **R2** from the roadmap *at the code level* before any DoE is spent.

---

#### Step 3 — Shared-grid DoE + persisted dataset (≈ 2–3 days)

*Fixes C4, H1, M3, M6, and unblocks everything downstream.*

1. Decide the MVP grid (roadmap §7.1 = 15 cases is the target):
   - Conditions: `{healthy, crack, misalignment}`
   - Shared speeds: `{650, 750, 900 rpm}` (extend crack to 900 rpm; extend misalignment to 650 rpm; this is one-line changes in the existing simulators)
   - Severity: `{0.1, 0.3, 0.5}` for crack depth ratio, `{(0.5, 0.25), (1.0, 0.5), (1.5, 0.75) mm}` for misalignment (x, y)
   - This gives 3 × 3 × 3 = 27 cases — slightly larger than the MVP but cheap to run.
2. Put this in a driver script `02_simula/run_campaign.py` that:
   - Loads `sinha_rotor.toml` once.
   - Loops over the DoE using ROSS's `run_crack` / `run_misalignment` / `run_time_response` (one entry point per condition, but all three going through the same time grid and same `num_modes`).
   - Writes each case to `02_simula/results/campaign.h5` (HDF5) with groups `case_<uuid>`, datasets `t`, `x_probe`, `y_probe`, `x_disk`, `y_disk`, and attributes capturing every parameter + `ross.__version__` + timestamp.
3. Add a one-line smoke-test notebook that loads the first case from the HDF5 and plots it — this proves the persistence layer works before the heavy run.

Estimated runtime: ~5 min per case × 27 = ~2.5 h. Parallelizable with `joblib` when needed.

---

#### Step 4 — Feature matrix + discrimination figure (≈ 2–3 days)

*Fulfills roadmap §5.2 Phase 4, §3.1 "Critical" row 2–3, and retires **R2** with real data.*

1. Write `02_simula/features.py` with scalar extractors operating on a single time series `(t, x)`:
   - `bispectral_peak_ratio(x, fs, Ω)` — `|B(1X,1X)| / |B(1X,2X)|`.
   - `bicoherence_sum(x, fs, Ω, band)` — `Σ b²` over a rectangular band around the (1X,1X)–(2X,2X) patch.
   - `bispectral_entropy(x, fs)` — `−Σ p log p` over the normalized bispectral magnitude.
   - `biphase(x, fs, Ω, (m, n))` — `angle(B(mΩ, nΩ))` in degrees.
2. A notebook `07_feature_matrix.ipynb` that: loads `campaign.h5`, calls the four extractors per case, builds a `pandas.DataFrame` `(N_cases × N_features + condition label)`, and saves it to `results/features.parquet`.
3. A discrimination figure (roadmap Fig. 13): 2-D scatter of `BPR` vs `bicoherence_sum`, markers colored by `condition`, size by severity. Expected outcome (per R2 mitigation): healthy clusters near origin, crack and misalignment separate on biphase or BPR axes. If they do not separate, add the trispectrum indicator before panicking — cubic coupling is the backup card the roadmap §9.2 already reserves.

---

#### Step 5 — Sensitivity curves + comparison to Sinha (≈ 1 week, runs in parallel with writing)

*Closes the "Quantitative validation against Sinha" row of the executive summary.*

1. Sensitivity curves (roadmap Figs. 10–12): indicator vs crack depth at fixed speed; indicator vs speed at fixed depth; indicator vs unbalance phase (you already have this sweep in `02`, just plug the HOS indicators in).
2. Qualitative comparison: render one `b²(f1, f2)` map for crack-severe @ 650 rpm alongside Sinha (2007) Fig. ~5–6. Shape, not amplitude, is what the reviewer will look at.
3. Draft the Methods section from the `signal_utils` docstrings — they become the paper's equations.

### 4.2 What to explicitly *not* do right now

To protect the 4-week window:

- **Do not** re-enable `Flex Open` / `Flex Breathing` crack models. They are a notebook `03` extension; the publishable MVP only needs Gasch or Mayes.
- **Do not** start on the RK4 experimental plan (`01_article/04_rk4_working_plan.md`) until the numerical HOS pipeline and discrimination figure exist. The RK4 extension is a big-paper win but it cannot substitute for Steps 1–4.
- **Do not** refactor the existing `01`–`04` notebooks beyond what Step 2 forces (the `signal_utils` module). They are already adequate Phase-2 artifacts; polishing them costs time that should go to Phase 4.

## 5. Pinned reference — the parameters this audit was performed against

If this notebook is reopened months from now, the cells below print the live `constants.py` values so the audit remains anchored. If the numbers diverge from those in Section 2.1, the audit is out of date.

In [3]:
from constants import (
    BEARING_1_NODE, BEARING_2_NODE, DISK_NODE, CRACK_NODE, PROBE_NODE,
    UNB_MAG, UNB_PHASE,
    SPEED_CRACK_0, SPEED_CRACK_1, SPEED_MIS_0, SPEED_MIS_1, SPEEDS,
    CRACK_RATIO, MIS_X, MIS_Y,
    DT, T, FS_SIM_HZ, FREQ_RANGE,
    SINHA_FS_HZ, SINHA_AA_CUTOFF_HZ,
    SINHA_HOS_DF_HZ, SINHA_HOS_N_SEGMENTS, SINHA_HOS_OVERLAP,
)
import numpy as np

print("Nodes:")
print(f"  bearings : {BEARING_1_NODE}, {BEARING_2_NODE}")
print(f"  disk/crack/probe : {DISK_NODE}, {CRACK_NODE}, {PROBE_NODE}")
print()
print("Forcing:")
print(f"  UNB_MAG   = {UNB_MAG}")
print(f"  UNB_PHASE = {UNB_PHASE}  ({np.rad2deg(UNB_PHASE.m):.1f} deg)")
print()
print("Speeds (rpm):")
print(f"  crack  : {SPEED_CRACK_0}, {SPEED_CRACK_1}")
print(f"  mis    : {SPEED_MIS_0}, {SPEED_MIS_1}")
print()
print("Severity:")
print(f"  CRACK_RATIO = {CRACK_RATIO}")
print(f"  MIS_X / MIS_Y = {MIS_X}, {MIS_Y}")
print()
print("Time / frequency grid:")
print(f"  DT = {DT}  =>  FS_SIM_HZ = {FS_SIM_HZ}  (Nyquist {FS_SIM_HZ/2} Hz)")
print(f"  T  length = {len(T)} samples, duration {T[-1]:.3f} s")
print(f"  FREQ_RANGE = {FREQ_RANGE}")
print()
print("Sinha (2007) HOS handoff:")
print(f"  fs = {SINHA_FS_HZ} Hz, AA cutoff = {SINHA_AA_CUTOFF_HZ} Hz")
print(f"  df = {SINHA_HOS_DF_HZ} Hz, N_seg = {SINHA_HOS_N_SEGMENTS}, overlap = {SINHA_HOS_OVERLAP}")
nfft_required = SINHA_FS_HZ / SINHA_HOS_DF_HZ
samples_required = nfft_required * (1 + SINHA_HOS_OVERLAP * (SINHA_HOS_N_SEGMENTS - 1))
print(f"  -> Nfft needed = {nfft_required:.0f}, total samples = {samples_required:.0f} "
      f"(~{samples_required / SINHA_FS_HZ:.1f} s at Sinha fs)")
print(f"  Current record provides only {len(T)} samples "
      f"(shortfall factor {samples_required / len(T):.1f}x).")

Nodes:
  bearings : 1, 11
  disk/crack/probe : 6, 7, 10

Forcing:
  UNB_MAG   = 0.0002 kilogram * meter
  UNB_PHASE = 4.1887902047863905 radian  (240.0 deg)

Speeds (rpm):
  crack  : 650 revolutions_per_minute, 750 revolutions_per_minute
  mis    : 750 revolutions_per_minute, 900 revolutions_per_minute

Severity:
  CRACK_RATIO = 0.5
  MIS_X / MIS_Y = 0.001 meter, 0.0005 meter

Time / frequency grid:
  DT = 0.001  =>  FS_SIM_HZ = 1000.0  (Nyquist 500.0 Hz)
  T  length = 2000 samples, duration 1.999 s
  FREQ_RANGE = [  0 200] Hz

Sinha (2007) HOS handoff:
  fs = 2560 Hz, AA cutoff = 1000 Hz
  df = 1.25 Hz, N_seg = 50, overlap = 0.5
  -> Nfft needed = 2048, total samples = 52224 (~20.4 s at Sinha fs)
  Current record provides only 2000 samples (shortfall factor 26.1x).


In [4]:
import ross as rs

rotor = rs.Rotor.load("sinha_rotor.toml")

assert rotor.disk_elements[0].n == DISK_NODE, "disk node drifted"
assert rotor.bearing_elements[0].n == BEARING_1_NODE
assert rotor.bearing_elements[1].n == BEARING_2_NODE

modal = rotor.run_modal(speed=0)
f1_hz = float(modal.wn[0] / (2 * np.pi))
print(f"ROSS version: {rs.__version__}")
print(f"ndof         : {rotor.ndof} ({rotor.number_dof} DOF per node)")
print(f"f1 (Hz)      : {f1_hz:.4f}   (target: 27.50)")
print(f"|Δf1|        : {abs(f1_hz - 27.5):.4f} Hz")

ROSS version: 2.2.0
ndof         : 78 (6 DOF per node)
f1 (Hz)      : 27.5000   (target: 27.50)
|Δf1|        : 0.0000 Hz


---

*Audit performed 2026-04-23 against commit `75a13b3`. Update this date and recompute the pinned cells before citing any figure in the thesis.*